In [1]:
import keras_tuner
import pandas as pd
import numpy as np
import sklearn
import os

from keras.src import callbacks
from keras.src.callbacks import ModelCheckpoint
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from ucimlrepo import fetch_ucirepo

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X_raw = adult.data.features
y_raw = adult.data.targets

KeyboardInterrupt: 

In [ ]:
X_raw.info()

In [ ]:
X_raw.describe(include='all')

In [ ]:
y_raw.value_counts()

In [ ]:
X_raw.isna().sum()

In [ ]:
X_raw['workclass'].value_counts()

In [ ]:
X_raw['occupation'].value_counts()

In [ ]:
X_raw['native-country'].value_counts()

In [ ]:
X_raw.groupby('education')['education-num'].nunique()
#Bekræfter at den ene er overflødig.

In [ ]:
X_raw.hist(bins=50, figsize=(18,12))

In [ ]:
count = y_raw['income'].value_counts()
count.plot(kind='pie')#Der er 4 target kategorier

In [ ]:
#Target har 4 kategorier, så vi fjerner de to med trailing . og samler dem.
y_clean = y_raw['income'].str.strip().str.rstrip('.')
y_clean.value_counts()

In [ ]:
#Vi laver ? om til NaN, fordi det ellers ville blive til en kategori når vi skal ohe det senere, hvilket ville
#være misvisende
X_clean = X_raw.replace('?', np.nan)
X_clean.isna().sum()
#Før havde vi både ? og NaN, de er nu alle NaN

X_clean = X_clean.drop(columns=['education'])
X_clean = X_clean.dropna()
y_clean = y_clean.loc[X_clean.index]

#grupper landende så vi har 10 ialt, da USA dominerer så meget.
top10_countries = X_clean['native-country'].value_counts().head(9).index #tager index op de 9 største lande
X_clean['native-country'] = X_clean['native-country'].where(X_clean['native-country'].isin(top10_countries), 'Other')
X_clean['native-country'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

#Vi splitter i tre sæt så vi har train til at fitte modellen, val til at fintune og test til at teste til sidst.
X_train_full, X_test, y_train_full, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

In [ ]:
#Stratify opdeler labels i lige store propertioner.
print(X_train.shape, X_val.shape, X_test.shape)
#normalize=true giver fordeling
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
X_clean.isna().any(axis=1).sum()
len(X_clean.dropna())

In [ ]:
#Encode
from sklearn.compose import ColumnTransformer
col_cat = ['workclass', 'occupation', 'marital-status', 'relationship', 'race', 'sex', 'native-country']
col_num = ['age', 'fnlwgt', 'education-num', 'hours-per-week', 'capital-gain', 'capital-loss']

preprocesser = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='if_binary'), col_cat)
    ,('num', StandardScaler(), col_num)
])

#drop=if-binary betyder at hvis featuren kun har 2 unike værdier ligesom sex,
#så dropper vi den ene kolonne, men dem med 3+ sker der ikke noget ved. De er redundante fordi er de perfekt anti correlated.

X_train = preprocesser.fit_transform(X_train)
X_val = preprocesser.transform(X_val)
X_test = preprocesser.transform(X_test)

y_train_binary = (y_train == '>50K').astype(int)
y_val_binary = (y_val == '>50K').astype(int)
y_test_binary = (y_test == '>50K').astype(int)


In [ ]:

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from keras_tuner import RandomSearch
import keras
keras.backend.clear_session()


In [ ]:
# Create a function that builds, compiles and returns a Keras model.
# Note that the function takes a HyperParameters object (hp) as a parameter, which it can
# use to define hyperparameters along with their range of possible values.

import keras_tuner as kt
import tensorflow as tf

def build_model(hp):
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2,
                             sampling="log")
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model = tf.keras.Sequential()
    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))
    model.add(tf.keras.layers.Dense(10, activation="sigmoidd"))
    model.compile(loss="binary_crossentropy", optimizer=optimizer,
                  metrics=["val_f1"])
    return model

In [ ]:
# Initializing Random Search
tuner = RandomSearch(
    build_model,
    objective=keras_tuner.Objective('val_f1', direction='max'),
    max_trials=20,  # number of combinations to try
    executions_per_trial=1,
    overwrite=True,
    directory='hyperparam_tuning',
    project_name='adult_model',
    seed=42
)

In [ ]:
#Callbacks
mc_cb = keras.callbacks.ModelCheckpoint(filepath="best_model.keras")
es_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, )
tb_cb = keras.callbacks.TensorBoard(log_dir='./logs')

In [ ]:
# Performing hyperparameter tuning
tuner.search(X_train, y_train_binary, epochs=30, validation_data=(X_val, y_val_binary), callbacks=[mc_cb, es_cb])

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
# Building the model with the best hyperparameters
model_hp = tuner.hypermodel.build(best_hp)

In [ ]:
import keras
import keras.metrics as km

model = keras.models.Sequential([
    #hidden layers
    keras.layers.Dense(100, activation='relu'),
    keras.layers.Dense(100, activation='relu'),
    keras.layers.Dense(100, activation='relu'),
    keras.layers.Dense(100, activation='relu'),
    #output layer
    keras.layers.Dense(1, activation='sigmoid')])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=[
        keras.metrics.Recall(),
        keras.metrics.Precision(),
        keras.metrics.F1Score(threshold=0.5, name='f1'),
    ])

In [ ]:
history = model_hp.fit(
    X_train, y_train_binary,
    validation_data=(X_val, y_val_binary),
    epochs=20,
    batch_size=32,
    callbacks=[es_cb, mc_cb],
)

In [ ]:
results = model_hp.evaluate(X_test, y_test_binary, return_dict=True)
for name, value in results.items():
    print(f"{name:10s}: {float(np.asarray(value).flatten()[0]):.4f}")